# unit04 レッスン: HTTP通信とスクレイピングのマナー

**このレッスンで作れるようになるもの**: サーバーからの応答(Response)を安全に読み解き、`robots.txt` で「取ってよいか」を判定し、送りすぎを防ぐレート制限と `429` 応答へのリトライを実装できる — 実務で「信頼できる書き方」と評価されるスクレイパーの土台。

スクレイピングは、**相手のサーバーの資源を借りてデータをもらう行為**です。技術的に取れることと、取ってよいことは別問題。`robots.txt` を無視したり、間隔を空けず大量にリクエストを送れば、**IPブロック・法務トラブル・最悪は相手の業務停止**という現実のリスクに直結します。逆に、この回で学ぶ「応答の正しい読み方・robots.txt の尊重・正直な名乗り・間隔を空ける・429 に従う」を押さえれば、多くの現場で通用します。

- 所要時間: 15〜25分
- 進め方: セルを上から順に実行(`Shift+Enter`)。「書いてみる」セルだけ自分で書く
- **ネットワークには一切繋ぎません**。`requests.get()` の戻り値そっくりのダミー応答(FakeResponse)とテキストで完結します
- 詰まったら: Claude に聞いてOK(答えではなくヒントをくれます)

In [ ]:
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


# --- 演習で使う FakeResponse をこのレッスンでも使えるように、同じものを定義しておく ---
# (data/fake_responses.py と同じ属性名・同じ振る舞い。cwd に依存せず動くよう、ここで定義し直す)
# 本物の requests.get(url) が返す Response オブジェクトのスタブ(替え玉)です。
# C# で HttpResponseMessage を単体テスト用にモックするのと同じ発想。
class FakeHTTPError(Exception):
    pass


class FakeResponse:
    # status_code / text / headers / ok は本物の requests.Response と同じ属性名
    def __init__(self, status_code, text="", headers=None):
        self.status_code = status_code      # HTTPステータス番号(200/404/429 など)
        self.text = text                    # 本文(HTMLなどの文字列)
        self.headers = headers or {}        # 応答ヘッダ(辞書のようにキーで引く)

    @property
    def ok(self):
        # 2xx/3xx なら成功とみなす(C#: response.IsSuccessStatusCode 相当)
        return 200 <= self.status_code < 400

    def raise_for_status(self):
        # 失敗応答なら例外を投げる(C#: response.EnsureSuccessStatusCode() 相当)
        if not self.ok:
            raise FakeHTTPError(f"HTTP error: {self.status_code}")


def make_ok_response(text):
    return FakeResponse(200, text=text, headers={"Content-Type": "text/html; charset=utf-8"})


def make_not_found_response():
    return FakeResponse(404, text="Not Found", headers={"Content-Type": "text/plain"})


def make_rate_limited_response(retry_after_seconds=5):
    return FakeResponse(
        429,
        text="Too Many Requests",
        headers={"Content-Type": "text/plain", "Retry-After": str(retry_after_seconds)},
    )


# このレッスンで題材にする robots.txt(演習の data/example_robots.txt と同じ内容)。
# 本物では requests でサイトの /robots.txt から取ってきた文字列がここに入る、と想像してください。
ROBOTS_TXT = """User-agent: *
Disallow: /admin/
Disallow: /private/
Crawl-delay: 2

User-agent: MyScraperBot
Disallow: /admin/
Allow: /private/reports/
Crawl-delay: 1

User-agent: BadBot
Disallow: /
"""

print("準備OK! FakeResponse と ROBOTS_TXT を用意しました。")
print("お試し: ", make_ok_response("<h1>hi</h1>").status_code, "/", make_rate_limited_response().status_code)

---
## 概念1: HTTPレスポンスの解剖 — status_code / text / headers / ok

### なぜ学ぶか
スクレイパーの入口は「サーバーに URL を投げて応答をもらう」ことです(パイプラインの [1 取得])。この応答には**成否**や**本文**や**付帯情報**が詰まっていて、それを読み間違えると「404 のエラーページを商品データとして解析してしまう」「文字化けした本文を保存してしまう」といった事故が起きます。まず応答オブジェクトを解剖して、どこに何が入っているかを頭に入れます。求人票の「API/Web からのデータ取得」はここが土台です。

### 解説

本物のスクレイピングでは `import requests` して `response = requests.get(url)` と書きます。この `response` は C# の `HttpClient.GetAsync()` が返す **`HttpResponseMessage` に相当**するオブジェクトで、主に次の4つを持ちます:

| 属性 | 何が入っているか | C# の対応 |
|------|------------------|-----------|
| `response.status_code` | HTTPステータス番号(整数) | `response.StatusCode` |
| `response.text` | 本文の文字列(HTMLなど) | `response.Content`(文字列化) |
| `response.headers` | 応答ヘッダ(辞書のように引く) | `response.Headers`(`Dictionary`) |
| `response.ok` | 成功したか(真偽値) | `response.IsSuccessStatusCode` |

**status_code の代表例**(3桁の番号でサーバーが結果を伝える):
- `200` = OK(成功。本文が使える)
- `404` = Not Found(そのURLは存在しない)
- `429` = Too Many Requests(送りすぎ。後で扱う)

`response.ok` は「2xx/3xx なら `True`」を返す便利属性です。`headers` は**辞書**なので、`headers.get("Content-Type", "unknown")` のように**キーが無くても安全に**取り出せます(unit01 でやった `dict.get`)。

このレッスンでは `requests` の代わりに、冒頭で定義した **`FakeResponse`(本物と同じ属性名を持つ替え玉)** を使います。演習コードは本物の Response を受け取っても FakeResponse を受け取っても**同じ書き方で動く**ように作られています。

In [ ]:
# GOAL: 応答オブジェクトの4つの属性(status_code / text / headers / ok)を目で見る

# STEP 1: 200 OK の応答を作る(本番なら requests.get(url) の戻り値に相当)
res = make_ok_response("<h1>本日のおすすめ焙煎豆</h1>")
print("status_code:", res.status_code)   # 200
print("ok         :", res.ok)             # True(2xxなので成功)
print("text        :", res.text)
print("headers     :", dict(res.headers))

# STEP 2: headers は辞書。get で安全に取り出す(無ければ既定値)
print("Content-Type:", res.headers.get("Content-Type", "unknown"))
print("無いヘッダ  :", res.headers.get("X-Nope", "unknown"))

# STEP 3: 失敗応答(404)だと ok がどうなるか
missing = make_not_found_response()
print("---")
print("404 の status_code:", missing.status_code, "/ ok:", missing.ok)

### 予測してみよう

次のセルは `429`(送りすぎ)の応答を作り、その `ok` を表示します。

**実行する前に予測**: `429` のとき `ok` は `True` と `False` のどちらでしょう?(ヒント: `ok` は「2xx/3xx なら成功」。`429` は 4xx です)

In [ ]:
# 予測してから実行!
limited = make_rate_limited_response(retry_after_seconds=5)
print("429 の status_code:", limited.status_code)
print("429 の ok         :", limited.ok)
print("Retry-After ヘッダ:", limited.headers.get("Retry-After"))

`429` は 4xx なので `ok` は `False`。そして `Retry-After` ヘッダで「何秒待てばいいか」が付いてくる — これが概念3で効いてきます。

### 書いてみる

**課題**: 下の `ok_res`(200 の応答)から、**成功しているなら本文 `text` を、そうでなければ `None` を**取り出して `result1` に入れてください(期待値: `"<p>data</p>"`)。

これは演習 ex01 の1問目 `get_body_if_ok` の中身そのものです。C# で言えば `response.IsSuccessStatusCode ? response.Content : null`。

ヒント(概念レベル): `ok_res.ok` が `True` なら `ok_res.text`、そうでなければ `None`。`if/else` で書いても、1行の条件式 `ok_res.text if ok_res.ok else None` でもOK。

In [ ]:
ok_res = make_ok_response("<p>data</p>")

result1 = None
# ここに書く(result1 に代入する。ok_res.ok を見て text か None を選ぶ)


check("概念1: 成功なら本文", result1, "<p>data</p>",
      hint="result1 = ok_res.text if ok_res.ok else None。ok_res.ok は True なので text が入る")

---
## 概念2: robots.txt — 「ここは取らないで」の紳士協定を読む

### なぜ学ぶか
`robots.txt` は、サイト運営者が「このパスはクロールしないでほしい」と表明する、サイト直下の約束ごとのファイル(例: `https://example.com/robots.txt`)です。**法的拘束力は必ずしも無い**ものの、無視すればアクセス遮断や法務トラブルの火種になります。実務では**巡回を始める前に必ず robots.txt を読み、許可されたパスだけを取りに行く**のが最低ラインのマナー。ここを守っているかどうかで、スクレイパーの「信頼できる書き方」度が決まります。

### 解説

`robots.txt` は次のような**プレーンテキスト**です(冒頭で `ROBOTS_TXT` に入れてあります):

```
User-agent: *              ← 「全ボット向け」のルール(* はワイルドカード)
Disallow: /admin/          ← /admin/ 以下は取るな
Crawl-delay: 2             ← リクエスト間隔は2秒空けろ

User-agent: MyScraperBot   ← 「MyScraperBot という名乗り向け」の個別ルール
Allow: /private/reports/   ← ここは特別に許可
```

自力でパースする必要はありません。Python の**標準ライブラリに `urllib.robotparser.RobotFileParser` というルールエンジンが最初から入っています**(C# には標準では無いので手実装かライブラリ導入が要る所)。使い方は3手:

1. `rp = RobotFileParser()` — 空のパーサを作る
2. `rp.parse(行のリスト)` — robots.txt の**各行のリスト**を渡して解析。
   ※ `rp.read()` というメソッドもありますが**これはネットにアクセスして取りに行く**ので、今回は使いません。手元の文字列を `text.splitlines()`(文字列を行ごとのリストに分割。C# の `text.Split('\n')` 相当)で渡します。
3. `rp.can_fetch(user_agent, path)` — 「この名乗りでこのパスを取ってよいか」を **`True`/`False`** で返す(C#風に言えば `rp.CanFetch(...)`)

さらに `rp.crawl_delay(user_agent)` で**その名乗り向けの待つべき秒数**を取れます(指定が無ければ `None`)。

In [ ]:
# GOAL: robots.txt をパースし、User-Agent 別に「取ってよいか」を判定する

from urllib.robotparser import RobotFileParser

# STEP 1: パーサを作り、robots.txt の中身を「行のリスト」にして渡す
rp = RobotFileParser()
rp.parse(ROBOTS_TXT.splitlines())   # splitlines() で1行=1要素のリストにする
print("パース完了")

# STEP 2: can_fetch で許可判定。同じパスでも User-Agent で結果が変わる
print("MyScraperBot /products/ :", rp.can_fetch("MyScraperBot", "/products/"))  # True
print("MyScraperBot /admin/    :", rp.can_fetch("MyScraperBot", "/admin/"))     # False(Disallow)
print("BadBot       /products/ :", rp.can_fetch("BadBot", "/products/"))        # False(全部Disallow)

# STEP 3: crawl_delay で「待つべき秒数」を取る(名乗りごとに違う)
print("---")
print("MyScraperBot の待ち秒数:", rp.crawl_delay("MyScraperBot"))   # 1
print("その他ボットの待ち秒数 :", rp.crawl_delay("SomeOtherBot"))    # 2(* のルールが適用)

### 予測してみよう

次のセルは、`MyScraperBot` が `/private/reports/monthly.html` を取ってよいかを判定します。`ROBOTS_TXT` をもう一度見てください: `*` 向けは `Disallow: /private/` ですが、`MyScraperBot` 向けには `Allow: /private/reports/` があります。

**実行する前に予測**: `MyScraperBot` のとき結果は `True` / `False` どちらでしょう?(名乗り専用ルールがある場合、`*` のルールと個別ルールのどちらが優先されるか)

In [ ]:
# 予測してから実行!
path = "/private/reports/monthly.html"
print("MyScraperBot:", rp.can_fetch("MyScraperBot", path))   # 個別ルールに Allow がある
print("SomeOtherBot:", rp.can_fetch("SomeOtherBot", path))   # * ルールの Disallow /private/ に当たる

名乗り専用ルールがあると、そちらが優先されます。だから**正直に名乗る**ことには「その名乗り向けの許可・間隔を正しく受け取れる」という実利もあるわけです(名乗りの話は概念3で)。

### 書いてみる

**課題**: 上で作ったパーサ `rp` を使い、**`BadBot` が `/products/` を取ってよいか**の判定結果(`True`/`False`)を `result2` に入れてください(期待値: `False` — `BadBot` は `Disallow: /` で全部禁止)。

これは演習 ex02 の `is_allowed` の中身そのものです。

ヒント(概念レベル): `rp.can_fetch("名乗り", "パス")` を呼ぶだけ。1行で書けます。

In [ ]:
result2 = None
# ここに書く(result2 に代入する。rp.can_fetch を BadBot と "/products/" で呼ぶ)


check("概念2: robots許可判定", result2, False,
      hint='result2 = rp.can_fetch("BadBot", "/products/")。BadBot は Disallow: / なので False')

---
## 概念3: レート制限と 429 リトライ — 「間隔を空ける」「言われたら待つ」

### なぜ学ぶか
たとえ robots.txt で許可されていても、**休みなく連射すれば相手のサーバーに負荷をかける迷惑行為**です。多くのサイトは間隔が短すぎると `429 Too Many Requests` を返して「送りすぎ、少し待って」と伝えてきます。実務で信頼されるスクレイパーは、(1) 各リクエストの間に**間隔を空け**、(2) `429` が来たら**素直に待ってからやり直す**の2点を必ず備えています。これは unit06 のキャップストーンで実サイト相当を巡回するときに、そのまま組み込む部品になります。

### 解説

**(1) 間隔を空ける(レート制限)**
「前回のリクエストから `min_interval` 秒たっていなければ、足りない分だけ待つ」。本番では `import time` して `time.sleep(秒数)` で待ちます。

ただしこのレッスン(と演習)では、**`time.sleep` を直接呼ばず、待つ処理を関数として外から渡します**。理由は2つ: 学習中に本当に数秒止まると体験が悪いこと、そして「**何秒待とうとしたか**」を検証したいこと。これは C# で `HttpClient` に自前のリトライ処理を差し込むときの発想(依存性注入 / デリゲート差し替え)と同じです。渡す関数を `sleep_func` と呼びます(本番は `time.sleep`、テストでは記録するだけのダミー)。

**(2) 429 が来たら待ってリトライ**
`429` 応答には多くの場合 `Retry-After` ヘッダで「何秒待て」が入っています。だから流れはこう:

```
if response.status_code == 429:
    wait = float(response.headers.get("Retry-After", 既定秒))  # ヘッダが無ければ既定値
    sleep_func(wait)   # その秒数だけ待つ
    # → もう一度取りに行く(リトライ)
```

`Retry-After` の値は**文字列**なので `float(...)` で数値に直します(unit01 の型変換)。ヘッダが無い場合に備えて `dict.get(キー, 既定値)` で受けるのが定石です。

**(3) 指数バックオフ(考え方だけ)**
リトライしても失敗が続くときは、待ち時間を `1秒 → 2秒 → 4秒 …` と倍々に伸ばす「指数バックオフ」という定石があります(C# の Polly ライブラリが有名)。**ただし実務でまず最初にやるべきは、難しいことより「1秒でいいから間隔を空ける」「正直な User-Agent を名乗る」**の2つ。ここを外さないのが一番効きます。

#### 補足: 正直な User-Agent(名乗り)

`User-Agent` はリクエスト時に「自分が何者か」を伝える文字列です。ブラウザのふりをして自分を偽るのではなく、**Bot名と連絡先を正直に書く**のがマナー:

```python
headers = {"User-Agent": "MyScraperBot/1.0 (+mailto:you@example.com)"}
response = requests.get(url, headers=headers)   # ← 本番の書き方(今回は実行しません)
```

こう名乗っておくと、相手が robots.txt であなた向けの許可・Crawl-delay を用意している場合に正しく適用され、問題があれば連絡ももらえます。

In [ ]:
# GOAL: (1) 間隔が足りなければ不足分だけ待つ (2) 429なら Retry-After 秒だけ待ってリトライ判定

# 検証用の「記録するだけの sleep」。実際には待たず、待とうとした秒数を集める。
waited = []
def fake_sleep(seconds):
    waited.append(seconds)   # 本番の time.sleep の代わり。何秒待つ「はず」だったかを記録

# STEP 1: 前回から min_interval 秒たっていないとき、不足分を計算して待つ
def wait_if_needed(now, last_called_at, min_interval, sleep_func):
    elapsed = now - last_called_at        # 経過時間
    remaining = min_interval - elapsed    # あと何秒待つ必要があるか
    if remaining > 0:
        sleep_func(remaining)
        return remaining
    return 0

# 前回0.0秒、今1.5秒、間隔2秒 → あと0.5秒待つべき
w = wait_if_needed(now=1.5, last_called_at=0.0, min_interval=2.0, sleep_func=fake_sleep)
print("待った秒数:", w, "/ 記録:", waited)

# STEP 2: 十分時間がたっていれば待たない(0を返し、記録も増えない)
w2 = wait_if_needed(now=5.0, last_called_at=0.0, min_interval=2.0, sleep_func=fake_sleep)
print("待った秒数:", w2, "/ 記録:", waited)

# STEP 3: 429応答に対して Retry-After 秒だけ待つ
res429 = make_rate_limited_response(retry_after_seconds=5)
retry_after = float(res429.headers.get("Retry-After", 1.0))   # 文字列 "5" を数値に
fake_sleep(retry_after)
print("Retry-After に従って待った:", retry_after, "秒 / 記録:", waited)

### 予測してみよう

次のセルは、`Retry-After` ヘッダを**持たない**別の 429 応答に対して待ち秒数を決めます。`headers.get("Retry-After", default)` で既定値 `1.0` を渡しています。

**実行する前に予測**: このとき `wait` は何秒になるでしょう?(ヒント: ヘッダが無いときの `dict.get` の戻り値)

In [ ]:
# 予測してから実行!
# Retry-After ヘッダを付けずに 429 を作る
res_no_header = FakeResponse(429, text="Too Many Requests", headers={})
wait = float(res_no_header.headers.get("Retry-After", 1.0))
print("Retry-After が無いときの待ち秒数:", wait)

ヘッダが無ければ既定値の `1.0` 秒。だから `dict.get(キー, 既定値)` で受けておくと、ヘッダの有無にかかわらず落ちません。

### 書いてみる

**課題**: `Retry-After: 5` の 429 応答 `res_limited` から、**待つべき秒数**を取り出して `result3` に入れてください(期待値: `5.0`)。ヘッダの値は文字列なので数値に直します。

これは演習 ex03 の `handle_rate_limit` が「何秒待つか」を決める部分そのものです。

ヒント(概念レベル): `res_limited.headers.get("Retry-After", 1.0)` で取り出し、`float(...)` で数値化。

In [ ]:
res_limited = make_rate_limited_response(retry_after_seconds=5)

result3 = None
# ここに書く(result3 に代入する。Retry-After ヘッダを取り出して float に変換)


check("概念3: 待つべき秒数", result3, 5.0,
      hint='result3 = float(res_limited.headers.get("Retry-After", 1.0))。ヘッダ値 "5" が 5.0 になる')

### もう一問(発展): 間隔が足りないとき、何秒待つべきか

**課題**: 前回リクエストが `0.0` 秒、今が `1.5` 秒、最低間隔が `2.0` 秒だとします。**あと何秒待つべきか**(不足分)を計算して `result4` に入れてください(期待値: `0.5`)。

これは演習 ex03 の `wait_if_needed` が待ち時間を決める計算そのものです。

ヒント(概念レベル): 「最低間隔 − 経過時間」。経過時間は `今 − 前回`。`2.0 - (1.5 - 0.0)` を計算するだけ。

In [ ]:
now = 1.5
last_called_at = 0.0
min_interval = 2.0

result4 = None
# ここに書く(result4 に代入する。min_interval から経過時間を引いた「不足分」)


check("概念3: 待つべき不足分", result4, 0.5,
      hint="経過時間 = now - last_called_at。不足分 = min_interval - 経過時間 = 2.0 - (1.5 - 0.0)")

---
## 振り返り(1〜2文でOK — このセルを編集して書き込んでください)

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:

(この記述はセッション終了時にチューターが学習ノートとスキルレベル判定に使います)

## まとめと次へ

| 概念 | 一言で | C#で言うと |
|------|--------|-----------|
| レスポンスの解剖 | `status_code`(200/404/429)・`text`・`headers`・`ok` で応答を読む | `HttpResponseMessage` / `IsSuccessStatusCode` |
| raise_for_status | 失敗応答なら例外を投げる | `EnsureSuccessStatusCode()` |
| robots.txt | `RobotFileParser` で `can_fetch` 判定・`crawl_delay` 取得。取る前に必ず確認 | 標準では無い(手実装/ライブラリ) |
| User-Agent | Bot名と連絡先を正直に名乗る | リクエストヘッダの設定 |
| レート制限 / 429 | 間隔を空ける・`Retry-After` に従って待ってリトライ。まず「1秒待つ+正直に名乗る」 | Polly のリトライ/バックオフ |

**この先どこで使うか**:
- **演習 ex01〜ex04** で、今日の各関数をスケルトンの TODO として自分で書き上げます(今日は各概念の一歩手前まで一緒にやりました)。
- **unit06 のキャップストーン**では、この「robots.txt を確認 → 取得 → 間隔を空ける → 429 なら待ってリトライ」の一連を、複数ページの巡回にそのまま組み込みます。今日作った `is_allowed` / `handle_rate_limit` / `wait_if_needed` が、実サイト相当を礼儀正しく巡回するスクレイパーの心臓部になります。

**次**: 演習 `ex01_parse_response.py` へ。lesson を見ながらで OK。テストは
`python -m pytest courses/web-scraping/unit04-http-and-manners/tests/test_ex01.py -q`